## Gold — `dim_area` (perfil das áreas declaradas)

**Origem:** `workspace.silver.cno_areas` → **Destino:** `workspace.gold.dim_area`

- **Modelo:** Star Schema. As colunas categóricas de `cno_areas` (`categoria`, `destinacao`, `tipo_de_obra`, `tipo_de_area`, `tipo_de_area_complementar`) viram uma dimensão de combinações; a metragem fica como **medida** na fato. Como uma obra pode ter várias áreas, o registro da obra se repete na fato para cada área/tipo — a fato referencia esta dimensão via `sk_area`.
- **Grão:** 1 linha por combinação distinta dos 5 códigos (dimensão de combinação, padrão Kimball para atributos que andam juntos).
- **Transformações:**
  - Distinct das combinações de códigos.
  - Colunas `*_descricao` decodificadas via domínios oficiais RFB.
  - `tipo_de_area_complementar` pode ser nulo (quando `tipo_de_area = P`); a fato usa join null-safe (`eqNullSafe`) nesta coluna.
  - `sk_area` = surrogate key sequencial ordenada pelos códigos (determinística).
- **Linhagem:** CSV dados.gov.br → `bronze.cno_areas` → `silver.cno_areas` → `gold.dim_area`.

In [0]:
%run ./_setup_cno

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments
from metadata.metadata import (
    DIM_AREA_COMMENTS,
    DOMINIO_CATEGORIA,
    DOMINIO_DESTINACAO,
    DOMINIO_TIPO_DE_OBRA,
    DOMINIO_TIPO_DE_AREA,
    DOMINIO_TIPO_DE_AREA_COMPLEMENTAR,
)

In [0]:
SOURCE_TABLE = "workspace.silver.cno_areas"
TARGET_TABLE = "workspace.gold.dim_area"

# Chaves da combinação (grão da dimensão)
COLUNAS_COMBINACAO = [
    "categoria",
    "destinacao",
    "tipo_de_obra",
    "tipo_de_area",
    "tipo_de_area_complementar",
]

# Contrato de saída gold.dim_area: código + descrição intercalados
COLUNAS_ORDENADAS = [
    "sk_area",
    "categoria",
    "categoria_descricao",
    "destinacao",
    "destinacao_descricao",
    "tipo_de_obra",
    "tipo_de_obra_descricao",
    "tipo_de_area",
    "tipo_de_area_descricao",
    "tipo_de_area_complementar",
    "tipo_de_area_complementar_descricao",
]

In [0]:
df = spark.table(SOURCE_TABLE).select(*COLUNAS_COMBINACAO).distinct()
print(f"Combinações distintas na silver: {df.count():,}")


def decodificar(coluna, dominio):
    """Gera expressão de descrição a partir do domínio oficial (código -> texto)."""
    expr = None
    for codigo, descricao in dominio.items():
        cond = F.col(coluna) == codigo
        expr = F.when(cond, F.lit(descricao)) if expr is None else expr.when(cond, F.lit(descricao))
    return expr


df = (
    df.withColumn("categoria_descricao", decodificar("categoria", DOMINIO_CATEGORIA))
    .withColumn("destinacao_descricao", decodificar("destinacao", DOMINIO_DESTINACAO))
    .withColumn("tipo_de_obra_descricao", decodificar("tipo_de_obra", DOMINIO_TIPO_DE_OBRA))
    .withColumn("tipo_de_area_descricao", decodificar("tipo_de_area", DOMINIO_TIPO_DE_AREA))
    .withColumn(
        "tipo_de_area_complementar_descricao",
        decodificar("tipo_de_area_complementar", DOMINIO_TIPO_DE_AREA_COMPLEMENTAR),
    )
)

# Surrogate key determinística: ordenação pelas natural keys da combinação
w = Window.orderBy(*COLUNAS_COMBINACAO)
df = df.withColumn("sk_area", F.row_number().over(w).cast("int"))

df = df.select(*COLUNAS_ORDENADAS)
print(f"Gold: {df.count():,} combinações na dimensão")
display(df.limit(25))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    DIM_AREA_COMMENTS
)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos_sk = spark.table(TARGET_TABLE).select("sk_area").distinct().count()
distintos_nk = spark.table(TARGET_TABLE).select(*COLUNAS_COMBINACAO).distinct().count()
print(f"Total: {total:,} | SK distintos: {distintos_sk:,} | Combinações distintas: {distintos_nk:,}")
assert total == distintos_sk == distintos_nk, "Quebra de unicidade SK/combinação em dim_area"
display(spark.sql(f"SELECT tipo_de_area, tipo_de_area_descricao, count(*) AS qtd_combinacoes FROM {TARGET_TABLE} GROUP BY tipo_de_area, tipo_de_area_descricao ORDER BY tipo_de_area"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))